In [1]:
import torch
from tqdm.notebook import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


# Carregando os dados

In [3]:
from datasets import load_dataset

ds = load_dataset("ruanchaves/b2w-reviews01")

ds = ds['train'].remove_columns(
    ['submission_date', 'reviewer_id', 'product_id', 'product_name', 'product_brand', 'site_category_lv1',
     'site_category_lv2', 'recommend_to_a_friend', 'review_title', 'reviewer_gender', 'reviewer_state',
     'reviewer_birth_year'])

ds = ds.filter(lambda example: example['overall_rating'] is not None)
ds = ds.filter(lambda example: example['review_text'] is not None)

ds = ds.class_encode_column('overall_rating')

ds = ds.train_test_split(test_size=0.08, shuffle=True, seed=42, stratify_by_column='overall_rating')

ds['train'], ds['validation'] = ds['train'].train_test_split(test_size=0.09, shuffle=True, seed=42,
                                                             stratify_by_column='overall_rating').values()

ds

DatasetDict({
    train: Dataset({
        features: ['overall_rating', 'review_text'],
        num_rows: 108080
    })
    test: Dataset({
        features: ['overall_rating', 'review_text'],
        num_rows: 10328
    })
    validation: Dataset({
        features: ['overall_rating', 'review_text'],
        num_rows: 10690
    })
})

In [4]:
#@title Removendo caracteres especiais (mantendo acentos e pontuação)

import re

# Remove qualquer char que não seja letra latina/acentuada, dígito, pontuação comum ou espaço
regex = re.compile(r"[^\w\s\-\'\.\,\!\?\;\:\(\)À-ÿ]", re.UNICODE)

ds_train = ds['train'].map(lambda example: {'review_text': regex.sub('', example['review_text'])})
ds_val = ds['validation'].map(lambda example: {'review_text': regex.sub('', example['review_text'])})
ds_test = ds['test'].map(lambda example: {'review_text': regex.sub('', example['review_text'])})

Map:   0%|          | 0/108080 [00:00<?, ? examples/s]

Map:   0%|          | 0/10690 [00:00<?, ? examples/s]

Map:   0%|          | 0/10328 [00:00<?, ? examples/s]

In [5]:
ds_train

Dataset({
    features: ['overall_rating', 'review_text'],
    num_rows: 108080
})

In [6]:
from collections import Counter

labels = ds_train['overall_rating']
class_names = ds_train.features["overall_rating"].names

counts = Counter(labels)

# Exibir distribuição
total = len(labels)
for cls, count in sorted(counts.items()):
    print(f"Classe {cls}: {count} exemplos ({100 * count / total:.1f}%)")

Classe 0: 21471 exemplos (19.9%)
Classe 1: 6803 exemplos (6.3%)
Classe 2: 13403 exemplos (12.4%)
Classe 3: 26779 exemplos (24.8%)
Classe 4: 39624 exemplos (36.7%)


In [7]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

# Converter para tensor na ordem correta das classes
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

# RNNs

## Embedding dos dados

In [8]:
from huggingface_hub import hf_hub_download
from safetensors.numpy import load_file
import numpy as np

path = hf_hub_download(repo_id="nilc-nlp/glove-300d", filename="embeddings.safetensors")

data = load_file(path)
vectors = data["embeddings"]

vocab_path = hf_hub_download(repo_id="nilc-nlp/glove-300d", filename="vocab.txt")
with open(vocab_path, encoding="utf-8") as f:
    vocab = [w.strip() for w in f]

embedding_matrix = np.array(vectors)  # shape: (929602, 300)
word2emb = dict(zip(vocab, embedding_matrix))

In [9]:
def batch_text_to_embedding(batch):
    import re
    import numpy as np

    sequences = []
    for text in batch["review_text"]:
        tokens = re.findall(r'\b\w+\b', text.lower())
        vecs = [word2emb[t] for t in tokens if t in word2emb]
        if not vecs:
            vecs = [np.zeros(300)]  # fallback para texto vazio
        sequences.append(np.array(vecs, dtype=np.float32))  # shape: (seq_len, 300)

    return {"embedding": sequences}

In [10]:
from datasets import load_from_disk
import os
from datasets import disable_caching

disable_caching()

if os.path.exists("data/train_embed"):
    train_embed = load_from_disk("data/train_embed")
else:
    train_embed = ds_train.map(
        batch_text_to_embedding,
        batched=True,
        batch_size=512,
        remove_columns=["review_text"],
        desc="Gerando embeddings de treino",
        keep_in_memory=True
    )
    train_embed.save_to_disk("data/train_embed")

if os.path.exists("data/val_embed"):
    val_embed = load_from_disk("data/val_embed")
else:
    val_embed = ds_val.map(
        batch_text_to_embedding,
        batched=True,
        batch_size=512,
        remove_columns=["review_text"],
        desc="Gerando embeddings de validação",
        keep_in_memory=True
    )
    val_embed.save_to_disk("data/val_embed")

if os.path.exists("data/test_embed"):
    test_embed = load_from_disk("data/test_embed")
else:
    test_embed = ds_test.map(
        batch_text_to_embedding,
        batched=True,
        batch_size=512,
        remove_columns=["review_text"],
        desc="Gerando embeddings de teste",
        keep_in_memory=True
    )
    test_embed.save_to_disk("data/test_embed")

Gerando embeddings de treino:   0%|          | 0/108080 [00:00<?, ? examples/s]

Saving the dataset (0/6 shards):   0%|          | 0/108080 [00:00<?, ? examples/s]

Gerando embeddings de validação:   0%|          | 0/10690 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10690 [00:00<?, ? examples/s]

Gerando embeddings de teste:   0%|          | 0/10328 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10328 [00:00<?, ? examples/s]

In [11]:
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence


def collate_fn(batch):
    inputs, labels = zip(*batch)
    # cada input é um tensor (seq_len, 300) — tamanhos diferentes, precisamos de padding
    inputs_padded = pad_sequence(inputs, batch_first=True, padding_value=0.0)
    # shape final: (batch, max_seq_len, 300)
    labels = torch.stack(labels)
    return inputs_padded, labels


class SequenceDataset(Dataset):
    def __init__(self, hf_dataset, label_col="label", embedding_col="embedding"):
        self.dataset = hf_dataset
        self.label_col = label_col
        self.embedding_col = embedding_col

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        embedding = torch.tensor(row[self.embedding_col], dtype=torch.float32)
        label = torch.tensor(row[self.label_col], dtype=torch.long)
        return embedding, label


train_ds = SequenceDataset(train_embed, label_col="overall_rating")
val_ds = SequenceDataset(val_embed, label_col="overall_rating")
test_ds = SequenceDataset(test_embed, label_col="overall_rating")

In [12]:
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, collate_fn=collate_fn, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, collate_fn=collate_fn, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, collate_fn=collate_fn, pin_memory=True)

In [13]:
#@title Funções de treino
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
from torch import nn
import copy


def accuracy_multiclass(outputs, labels):
    _, preds = torch.max(outputs, dim=1)
    if labels.dim() == 2 and labels.shape[1] == 1:
        labels = labels.squeeze(1)
    return torch.sum(preds == labels).item() / len(preds)


def validate_one_epoch(modelo, loss, val_loader: DataLoader, epoch_index: int, class_names):
    total_loss = 0.0
    total_accuracy = 0.0
    num_batches = len(val_loader)
    all_preds = []
    all_labels = []

    # desativamos o gradiente
    with torch.no_grad():
        pbar = tqdm(val_loader, desc=f"Epoch {epoch_index} (Validation)")

        # para cada batch de dados de validação
        for inputs, labels in pbar:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # fazemos uma predição e calculamos sua loss e acurácia
            outputs = modelo(inputs)
            l = loss(outputs, labels)

            _, preds = torch.max(outputs, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

            total_loss += l.item()
            total_accuracy += (preds == labels).float().mean().item()

            pbar.set_postfix(loss=f"{l.item():.4f}", acc=f"{(preds == labels).float().mean().item():.4f}")

    avg_loss = total_loss / num_batches
    avg_acc = total_accuracy / num_batches

    report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True, zero_division=0)

    return avg_loss, avg_acc, report


def train_one_epoch(modelo, loss, otimizador, train_loader: DataLoader, epoch_index: int, l2_lambda: float = 0.0,
                    verbose=False):
    total_loss = 0.0
    total_accuracy = 0.0
    num_batches = len(train_loader)

    if verbose:
        pbar = tqdm(train_loader, desc=f"Epoch {epoch_index} (Train)")
    else:
        pbar = train_loader

    # para cada batch de dados de treino
    for batch_idx, (inputs, labels) in enumerate(pbar):
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        otimizador.zero_grad()  # zeramos o gradiente para o batch
        outputs = modelo(inputs)  # fazemos uma predição

        # computamos a loss e seu gradiente (com regularização l2 opcional)
        l = loss(outputs, labels)
        if l2_lambda > 0:
            l += l2_lambda * sum(p.pow(2).sum() for p in modelo.parameters())  # regularização L2
        l.backward()

        # ajustamos o otimizador
        otimizador.step()

        # registramos a loss e a acurácia da batch
        current_batch_loss = l.item()
        with torch.no_grad():
            current_batch_accuracy = accuracy_multiclass(outputs, labels)

        if verbose:
            pbar.set_postfix(loss=f"{current_batch_loss:.4f}", acc=f"{current_batch_accuracy:.4f}")

        total_loss += current_batch_loss
        total_accuracy += current_batch_accuracy

    return total_loss / num_batches, total_accuracy / num_batches


def train(modelo, loss, otimizador, train_loader: DataLoader, epochs: int, class_names,
          val_loader: DataLoader | None = None,
          early_stopping_patience: int | None = None, l2_lambda: float = 0.0, verbose=False):
    log = {}

    if early_stopping_patience:
        if not val_loader:
            raise ValueError("Early stopping precisa de um conjunto de validação")
        last_val_loss = float("inf")
        current_patience = early_stopping_patience
        best_model_weights = copy.deepcopy(modelo.state_dict())

    for epoch_num in range(1, epochs + 1):
        log[epoch_num] = {}

        modelo.train(True)
        train_loss, train_accuracy = train_one_epoch(modelo, loss, otimizador, train_loader, epoch_num, l2_lambda,
                                                     verbose)

        log[epoch_num]["train_loss"] = train_loss
        log[epoch_num]["train_accuracy"] = train_accuracy

        if val_loader:
            modelo.eval()
            val_loss, val_accuracy, val_report = validate_one_epoch(modelo, loss, val_loader, epoch_num,
                                                                    class_names=class_names)

            log[epoch_num]["val_loss"] = val_loss
            log[epoch_num]["val_accuracy"] = val_accuracy
            log[epoch_num]["val_accuracy"] = val_accuracy
            log[epoch_num]["val_report"] = val_report

            if early_stopping_patience:
                if val_loss < last_val_loss:
                    last_val_loss = val_loss
                    current_patience = early_stopping_patience
                    best_model_weights = copy.deepcopy(modelo.state_dict())
                else:
                    current_patience -= 1
                    if current_patience == 0:
                        print("Early stopping")
                        break

    if early_stopping_patience and 'best_model_weights' in locals():
        modelo.load_state_dict(best_model_weights)
    return log

## Elman Network

In [14]:
class RNNClassifier(nn.Module):
    def __init__(self, input_size=300, hidden_size=150, num_classes=5, dropout=0.01):
        super().__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 64)
        self.activation1 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, num_classes)
        self.activation2 = nn.Softmax(dim=1)

    def forward(self, x):
        output, _ = self.rnn(x)
        x = output[:, -1, :]

        x = self.activation1(
            self.fc1(
                self.dropout1(x)
            )
        )
        x = self.activation2(
            self.fc2(
                self.dropout2(x)
            )
        )

        return x

In [15]:
elman_model = RNNClassifier().to(device)

otimizador = torch.optim.Adam(elman_model.parameters(), lr=0.003, weight_decay=0.0003)
loss = nn.CrossEntropyLoss(weight=class_weights_tensor)

display(elman_model)

RNNClassifier(
  (rnn): RNN(300, 150, batch_first=True)
  (dropout1): Dropout(p=0.01, inplace=False)
  (fc1): Linear(in_features=150, out_features=64, bias=True)
  (activation1): ReLU()
  (dropout2): Dropout(p=0.01, inplace=False)
  (fc2): Linear(in_features=64, out_features=5, bias=True)
  (activation2): Softmax(dim=1)
)

In [16]:
history = train(elman_model, loss, otimizador, train_loader, epochs=200, class_names=class_names, val_loader=val_loader,
                early_stopping_patience=10, verbose=True)

Epoch 1 (Train):   0%|          | 0/423 [00:00<?, ?it/s]

Epoch 1 (Validation):   0%|          | 0/42 [00:00<?, ?it/s]

Epoch 2 (Train):   0%|          | 0/423 [00:00<?, ?it/s]

Epoch 2 (Validation):   0%|          | 0/42 [00:00<?, ?it/s]

Epoch 3 (Train):   0%|          | 0/423 [00:00<?, ?it/s]

...

Epoch 6 (Validation):   0%|          | 0/42 [00:00<?, ?it/s]

Epoch 7 (Train):   0%|          | 0/423 [00:00<?, ?it/s]

Epoch 7 (Validation):   0%|          | 0/42 [00:00<?, ?it/s]

Epoch 8 (Train):   0%|          | 0/423 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
last_epoch = max(history.keys())
report = history[last_epoch]["val_report"]
print(f"F1-macro: {report['macro avg']['f1-score']:.4f}")

# Para ver a evolução do F1-macro ao longo das epochs:
f1_macro = [history[e]["val_report"]["macro avg"]["f1-score"] for e in history if "val_report" in history[e]]

## Treinando GRU

# Transformers

## Treinando Bertimbau

## Treinando Albertina